In [10]:
#Libraries constants and utilities
#Libraries
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import re

#Constants
DATA = Path("../data")

#Utilities
import sys
sys.path.append('../utilities')

from utils import (
    check_datasets_availability,
    missing_audit,
    compare_schemas,
    categorize_mood,
    analyze_mood_distribution,
    profile_numeric_columns
)
check_datasets_availability()

All datasets found, this notebook can be run without any issues.


# 2.Data Cleaning
This notebook applies the cleaning steps identified during profiling in `01-data profiling.ipynb` and will serve as a preparation for the Experiments the project aims to look into, along the way of cleaning some additional steps might be taken or revised. Cleaned datasets are saved as a new file under `../data/`. 

This notebook applies only corrections that are **data quality issues** (blunders, type conversions, standardization). Analytic choices such as outlier thresholds, variable encodings, and feature selection are deferred to the experiment notebooks where they can be documented with their hypothesis context.

## Analytic datasets expected output
| Output file | Source | Changes applied |
|---|---|---|
| `ds2_cleaned.csv` | `universal_top_spotify_songs.csv` | datetime conversion, year/month columns, global rows separated |
| `ds2_global_chart.csv` | same - due to fiel size | 28 908 null-country rows preserved separately |
| `ds3_cleaned.csv` | `mxmh_survey_results.csv` | 2 BPM blunders set to NaN, Permissions dropped, columns renamed |
| `ds4_with_codes.csv` | `World-happiness-report-2024.csv` | ISO-2 country_code column added |

### DS2 Top Spotify Songs in 73 Countries Cleaning
**Used in:** Experiment 1 (geographic + seasonal mood fingerprints) and Experiment 2 (national 
wellbeing vs. mood).

Cleaning steps:
1. Convert `snapshot_date` from string to `datetime64` - required for any temporal analysis
2. Extract `year` and `month` columns - Experiment 1 needs seasonal grouping and storing these as explicit columns avoids repeated parsing and makes the temporal dimension transparent
3. Separate the 28 908 null-country rows into a standalone file - they are valid chart data but cannot be assigned to any country
5. Verify no audio feature columns have missing values in the country-assigned set

In [3]:
ds2 = pd.read_csv(DATA / "universal_top_spotify_songs.csv")
print(f"Loaded: {ds2.shape[0]:,} rows, {ds2.shape[1]} columns")

#Converting snapshot_date to datetime
ds2["snapshot_date"] = pd.to_datetime(ds2["snapshot_date"])

# Extracting year and month as explicit columns
ds2["year"] = ds2["snapshot_date"].dt.year
ds2["month"] = ds2["snapshot_date"].dt.month

print("snapshot_date dtype:", ds2["snapshot_date"].dtype)
print(f"Date range: {ds2['snapshot_date'].min().date()} to {ds2['snapshot_date'].max().date()}")
print(f"Years covered: {sorted(ds2['year'].unique())}")

#Separating null-country rows - valid data, but not usable in country-level experiments
ds2_global = ds2[ds2["country"].isnull()].copy()
ds2_countries = ds2[ds2["country"].notna()].copy()

print(f"Country-assigned rows: {len(ds2_countries):,}")
print(f"Global chart rows (null country): {len(ds2_global):,}")
print(f"Accounts for total: {len(ds2_countries) + len(ds2_global):,} (original: {len(ds2):,})")

# Verifying all audio features are complete in the country-assigned set
AUDIO_FEATURES = ["valence", "energy", "tempo", "loudness", "danceability", "acousticness", "instrumentalness", "speechiness", "liveness"]

missing_check = ds2_countries[AUDIO_FEATURES].isnull().sum()
if missing_check.any():
    print("Missing audio features detected:")
    print(missing_check[missing_check > 0])
else:
    print("All audio feature columns complete - no missing values")

ds2_countries.head(3)

Loaded: 2,110,316 rows, 25 columns
snapshot_date dtype: datetime64[ns]
Date range: 2023-10-18 to 2025-06-11
Years covered: [np.int32(2023), np.int32(2024), np.int32(2025)]
Country-assigned rows: 2,081,408
Global chart rows (null country): 28,908
Accounts for total: 2,110,316 (original: 2,110,316)
All audio feature columns complete - no missing values


,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,year,month
50,7c5uGV9Rys18JP2570ykTu,Isaka (6am),"CIZA, Jazzworx, Thukuthela",1,0,0,ZA,2025-06-11,75,False,...,0,0.0566,0.29200,0.27900,0.2120,0.752,176.762,3,2025,6
51,73MJ65QLIfsU2GyCt5KQ3a,Uzizwa Kanjan (feat. GL_Ceejay),"Jazzworx, MaWhoo, Thukuthela, GL_Ceejay",2,0,0,ZA,2025-06-11,72,False,...,1,0.0518,0.58200,0.37100,0.1100,0.560,118.055,4,2025,6
52,4nP3JxLWurlrvollaDlxsJ,Mali,"Dlala Thukzin, Zee Nxumalo, SYKES",3,0,0,ZA,2025-06-11,69,False,...,0,0.0518,0.00606,0.00329,0.0526,0.291,118.050,4,2025,6


In [4]:
ds2_countries.to_csv(DATA / "ds2_cleaned.csv", index=False)
ds2_global.to_csv(DATA / "ds2_global_chart.csv", index=False)

### DS3 Music & Mental Health Survey Cleaning
**Used in:** Experiment 3 (mental health profiles vs. genre preferences)

Cleaning steps:
1. Set 2 BPM blunders to NaN - values of 999,999,999 and 624 are data entry errors (blunders in Stanford handbook terms), not statistical outliers. The rows are kept only the invalid BPM values are nulled. Whether to use BPM at all is an analytic decision for Experiment 3.
2. Drop the `Permissions` column - constant value "I understand." across all 736 rows, zero analytical information.
3. Rename all columns to snake_case without spaces or punctuation - good practice for reproducibility across tools
4. Genre frequency columns will actually be kept as original strings (Never / Rarely / Sometimes / Very frequently). Encoding them as ordered integers is an analytic choice that depends on the statistical method which will be used in Experiment 3 - it will be determined if needed there.

In [28]:
ds3 = pd.read_csv(DATA / "mxmh_survey_results.csv")
print(f"Loaded: {ds3.shape[0]} rows, {ds3.shape[1]} columns")

#Fixing the 2 BPM blunders
bpm_blunders = ds3[ds3["BPM"] > 300][["Timestamp", "Primary streaming service", "Age","Fav genre", "BPM"]]
print(f"BPM blunders to be nulled ({len(bpm_blunders)} rows):")
display(bpm_blunders)

ds3.loc[ds3["BPM"] > 300, "BPM"] = float("nan")
print(f"\nBPM values > 300 set to NaN. Remaining BPM missing: {ds3['BPM'].isnull().sum()}")

#Dropping Permissions column constant "I understand."
ds3 = ds3.drop(columns=["Permissions"])

#Renaming columns to snake_case
def to_snake(col: str) -> str:
    m = re.match(r'Frequency \[(.+)\]', col)
    if m:
        genre = m.group(1).lower().replace(' ', '_').replace('&', 'n')  # & → n
        return f'freq_{genre}'
    
    col = col.lower()
    col = re.sub(r'&', 'n', col)
    col = re.sub(r'[^a-z0-9\s]', '', col)
    col = re.sub(r'\s+', '_', col.strip())
    return col

ds3.columns = [to_snake(c) for c in ds3.columns]
print(ds3.columns.tolist())

#Verifying mental health scores are intact (double check never hurts)
MH_COLS = ["anxiety", "depression", "insomnia", "ocd"]
print("\nMental health score verification:")
print(f"Missing values: {ds3[MH_COLS].isnull().sum().sum()} (expected 0)")
print(f"All within 0-10: {((ds3[MH_COLS] >= 0) & (ds3[MH_COLS] <= 10)).all().all()}")

ds3.to_csv(DATA / "ds3_cleaned.csv", index=False)

Loaded: 736 rows, 33 columns
BPM blunders to be nulled (2 rows):


,Timestamp,Primary streaming service,Age,Fav genre,BPM
568,9/4/2022 15:41:59,Spotify,16.0,Video game music,999999999.0
644,9/13/2022 1:55:43,Other streaming service,16.0,EDM,624.0



BPM values > 300 set to NaN. Remaining BPM missing: 109
['timestamp', 'age', 'primary_streaming_service', 'hours_per_day', 'while_working', 'instrumentalist', 'composer', 'fav_genre', 'exploratory', 'foreign_languages', 'bpm', 'freq_classical', 'freq_country', 'freq_edm', 'freq_folk', 'freq_gospel', 'freq_hip_hop', 'freq_jazz', 'freq_k_pop', 'freq_latin', 'freq_lofi', 'freq_metal', 'freq_pop', 'freq_rnb', 'freq_rap', 'freq_rock', 'freq_video_game_music', 'anxiety', 'depression', 'insomnia', 'ocd', 'music_effects']

Mental health score verification:
Missing values: 0 (expected 0)
All within 0-10: True


### DS4 World Happiness Report - 2024 Cleaning
**Used in:** Experiment 1 (geographic + seasonal mood fingerprints) and Experiment 2 (national wellbeing vs. mood).

Cleaning steps from profiling:
1. Build an ISO 3166 alpha-2 country code to country name mapping for the 72 DS2 country codes. For the mapping this [IBAN table](https://www.iban.com/country-codes) will be used.
2. Add a `country_code` column to DS4 so it can be joined to DS2 in later experiments
3. Investigate the 3 countries with missing factor columns against DS2 coverage
4. Report effective join coverage (the 50.3% theoretical ceiling from profiling was a maximum estimate)

Note: country names in DS4 may not match the ISO standard names exactly. Manual overrides might be applied where needed in the dictionary.

In [38]:
ds4 = pd.read_csv(DATA / "World-happiness-report-2024.csv")

#Manually adjusted after - mannualy adjusted 2 country names US and TR
ISO2_TO_DS4_NAME = {
    "AE": "United Arab Emirates",
    "AR": "Argentina",
    "AT": "Austria",
    "AU": "Australia",
    "BE": "Belgium",
    "BG": "Bulgaria",
    "BO": "Bolivia",
    "BR": "Brazil",
    "BY": "Belarus",
    "CA": "Canada",
    "CH": "Switzerland",
    "CL": "Chile",
    "CO": "Colombia",
    "CR": "Costa Rica",
    "CZ": "Czechia",
    "DE": "Germany",
    "DK": "Denmark",
    "DO": "Dominican Republic",
    "EC": "Ecuador",
    "EE": "Estonia",
    "EG": "Egypt",
    "ES": "Spain",
    "FI": "Finland",
    "FR": "France",
    "GB": "United Kingdom",
    "GR": "Greece",
    "GT": "Guatemala",
    "HK": "Hong Kong S.A.R. of China",
    "HN": "Honduras",
    "HU": "Hungary",
    "ID": "Indonesia",
    "IE": "Ireland",
    "IL": "Israel",
    "IN": "India",
    "IS": "Iceland",
    "IT": "Italy",
    "JP": "Japan",
    "KR": "South Korea",
    "KZ": "Kazakhstan",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "LV": "Latvia",
    "MA": "Morocco",
    "MX": "Mexico",
    "MY": "Malaysia",
    "NG": "Nigeria",
    "NI": "Nicaragua",
    "NL": "Netherlands",
    "NO": "Norway",
    "NZ": "New Zealand",
    "PA": "Panama",
    "PE": "Peru",
    "PH": "Philippines",
    "PK": "Pakistan",
    "PL": "Poland",
    "PT": "Portugal",
    "PY": "Paraguay",
    "RO": "Romania",
    "SA": "Saudi Arabia",
    "SE": "Sweden",
    "SG": "Singapore",
    "SK": "Slovakia",
    "SV": "El Salvador",
    "TH": "Thailand",
    "TR": "Turkey",
    "TW": "Taiwan",
    "UA": "Ukraine",
    "US": "United States",
    "UY": "Uruguay",
    "VE": "Venezuela",
    "VN": "Vietnam",
    "ZA": "South Africa",
}

# Invert the mapping: DS4 name -> ISO code
DS4_NAME_TO_ISO2 = {v: k for k, v in ISO2_TO_DS4_NAME.items()}

ds4["country_code"] = ds4["Country name"].map(DS4_NAME_TO_ISO2)

matched = ds4["country_code"].notna().sum()
unmatched_names = ds4[ds4["country_code"].isna()]["Country name"].tolist()

print(f"DS4 countries matched to ISO code: {matched} of {len(ds4)}")
print(f"DS4 countries that could join to DS2: {matched}")
print(f"\nDS4 countries with no match (not present in DS2):")
for name in unmatched_names[:10]:
    print(f"  - {name}")
if len(unmatched_names) > 10:
    print(f"  ... and {len(unmatched_names) - 10} more")

DS4 countries matched to ISO code: 69 of 143
DS4 countries that could join to DS2: 69

DS4 countries with no match (not present in DS2):
  - Kuwait
  - Slovenia
  - Kosovo
  - Taiwan Province of China
  - Serbia
  - Malta
  - Uzbekistan
  - Cyprus
  - China
  - Bahrain
  ... and 64 more


In [44]:
#From profiling: Bahrain (BH), Tajikistan (TJ), and State of Palestine (PS) have missing audio features data
factor_cols = [
    "Log GDP per capita", "Social support", "Healthy life expectancy", "Freedom to make life choices", "Generosity", "Perceptions of corruption"
]
missing_factors = ds4[ds4[factor_cols].isnull().any(axis=1)][["Country name", "country_code", "Ladder score"]]
print("Countries with missing factor data:")
display(missing_factors)

#Even though it is visually present that country_code was not mapped - code validation:
ds2_codes = set(ds2_countries["country"].unique())
for _, row in missing_factors.iterrows():
    in_ds2 = row["country_code"] in ds2_codes if pd.notna(row["country_code"]) else False
    print(f"{row['Country name']}: present in DS2? {in_ds2}")

ds4.to_csv(DATA / "ds4_cleaned.csv", index=False)

Countries with missing factor data:


,Country name,country_code,Ladder score
61,Bahrain,NaN,5.959
87,Tajikistan,NaN,5.281
102,State of Palestine,NaN,4.879


Bahrain: present in DS2? False
Tajikistan: present in DS2? False
State of Palestine: present in DS2? False


# Cleaning Summary and Decisions
### Done in this notebook (data quality corrections only)
- Type conversions (`snapshot_date` to datetime)
- Standardization (year/month extraction, column renaming)
- Blunder correction (2 invalid BPM values set to NaN)
- Redundant column removal (Permissions - constant value)
- Cross-dataset join preparation (country_code mapping for DS2-DS4 join)

### Deliberately deferred to experiment notebooks as analytic choices
- Genre frequency encoding (Never/Rarely/Sometimes/Very frequently -> integer). This is an analytic choice that depends on the statistical test used in Experiment 3.
- Filtering DS2 to 2024 for Experiment 2 alignment. The right time window is a hypothesis-level decision.
- Thayer's mood category assignment (valence/energy thresholds). These thresholds will be calibrated and varied in Experiment 1.
- Any outlier treatment beyond the 2 confirmed BPM blunders.

In [46]:
#Safety net check to make sure that all cleaned datasets have been saved in the ../data folder
files = ["ds2_cleaned.csv", "ds2_global_chart.csv", "ds3_cleaned.csv", "ds4_cleaned.csv"]

for fname in files:
    fpath = DATA / fname
    if fpath.exists():
        df_check = pd.read_csv(fpath, nrows=1)
        print(f" {fname}: exists")
    else:
        print(f" {fname}: MISSING - something went wrong above")

 ds2_cleaned.csv: exists
 ds2_global_chart.csv: exists
 ds3_cleaned.csv: exists
 ds4_cleaned.csv: exists
